In [102]:
from collections import Counter
from combunipotent.tool import *
from multiset import Multiset

def multiset_diff(a,b):
    """
    Suppose a is a multiset and b is a set, this function returns
    a - b where b is viewed as a multiset with no duplication.
    """
    return Multiset(a) - Multiset(b)

"""
The following function generates all distinguished reduced marking 
"""
def allDistRmarkingBD(n):
    for lam in BDpartitions(n):
        nu  = sorted(set(multiset_diff(lam,set(lam))),reverse=True)
        if isBDDistingushed(nu,lam):
            yield (nu,lam)

def twogammaDistBD(nu,lam):
    assert(isBDDistingushed(nu,lam))
    part  = list_up(nu) + sorted(multiset_diff(lam,nu),reverse=True) 
    return twogamma_part(sum(lam)//2,part)
    

In [103]:
p = [7,8,8,9,9,5,3,2,2,1,1,1]
p.sort(reverse=True)
print(p)
print(MarkableBD(p))
print(isRMarkingBD([9,3],p))
print(isSpecialRMBD([7,3],p))

[9, 9, 8, 8, 7, 5, 3, 2, 2, 1, 1, 1]
{1, 5, 9}
False
True


In [104]:
for nu,lam in allDistRmarkingBD(12):
    print(f"dist. mpart. :  ({nu},{lam})")
    print(f"inf. char. ={twogammaDistBD(nu,lam)}/2")

dist. mpart. :  ([],[11, 1])
inf. char. =[10, 8, 6, 4, 2, 0]/2
dist. mpart. :  ([],[9, 3])
inf. char. =[8, 6, 4, 2, 2, 0]/2
dist. mpart. :  ([],[7, 5])
inf. char. =[6, 4, 2, 4, 2, 0]/2
dist. mpart. :  ([5, 1],[5, 5, 1, 1])
inf. char. =[5, 3, 1, 4, 2, 0]/2


In [105]:
tworhoplus(10)

[9, 7, 5, 3, 1]

In [106]:
list(list_up([5,4,3,1]))

[6, 3, 4, 0]

In [107]:
"""
To bootstrap, we need sgn repn of type B/C,D
as symbols. 
"""


def sgnBC(n):
    """
    Generate the symbol of sgn representation of type B/C
    """
    return (tuple(range(n+1)),tuple(range(1,n+1)))

def sgnD(n):
    """
    Generate the symbol of sgn representation of type D 
    TODO: check the correctness
    """
    return (tuple(range(n)),tuple(range(1,n+1)))

  

"""
Compute  the J-induction of a symbol 
       ( a_0, a_1, ... , a_k )
tau =  (   b_1, ... , b_k )

from a maximal parabolic of the form W_{A_r} x W_{r_0} 
which are the symbol  

       ( a_0, a_1, ... , a_i +1, ... a_k )
tau =  (   b_1, ... , b_i +1, ..., b_k )

We will not check tau is a valide symbol
"""
def JindPara(tau,r):
    tauL, tauR= equiv_symbol_expend(tau,r)
    hr = r//2
    if hr == 0:
        tauL1,tauL2 = list(tauL), []
        tauR1,tauR2 = list(tauR), []
    else: 
        tauL1,tauL2 = tauL[:-hr], tauL[-hr:]
        tauR1,tauR2 = tauR[:-hr], tauR[-hr:]

    #print(f"({tauL1},{tauR1})")
    #print(f"({tauL2},{tauR2})")
    if r %2 ==0:
        yield (tauL1+[a+1 for a in tauL2],tauR1 + [a+1 for a in tauR2])
    else:
        yield (tauL1[:-1] +[tauL1[-1]+1] + [a+1 for a in tauL2],tauR1 + [a+1 for a in tauR2])
        if len(tauR1) >0 and tauL1[-1]==tauR1[-1]:
            yield (tauL1 + [a+1 for a in tauL2],tauR1[:-1] + [tauR1[-1]+1]+ [a+1 for a in tauR2])


def Jind(ltau, lr):
    """
    ltau is a list of a left cell
    lr is a list [r_1, r_2, ... r_k]
    The return is Jind_{W_A_r_1 x W_A_r_2 x ... W_A_r_k} ltau
    """
    if len(lr)==0 or lr[0]==0:
        return ltau
    return Jind([tau2 for tau1 in ltau for tau2 in JindPara(tau1,lr[0])],lr[1:])





In [108]:
"""
Fix an integral infinitesimal character gamma,
Let W_gamma be the stablizer of gamma in W.
Let W(gamma) be the integral Weyl group of gamma.
the Lusztig cell is given by 
J-ind_{W_gamma}^{W(gamma)} (sgn) \otimes sgn
"""


"""
Let G = type BCD
If gamma = (g_1 = g_2 = ... = g_k_1 > g_{k_1+1}  = g_{k_1+2} = g_{k_1+k_2} > g_{k_1+k_2+1} = ... = g_{k_1+k_2+k_3} > 0 = 0 = ... = 0) 
Assume there are k_0 zeros in gamma
then W_gamma = W_{A_k_1} x W_{A_k_2} x W_{A_k_3} x ... x W_{k_r} x W_{k_0,BCD}

Wgamma send gamma to the tuples 
((k_0, [k'_1,k'_2,...,k'_r']), # of even parts 
   (0, [k''_1,k''_2,...,k''_r''])) # of odd parts
"""
def Wgamma(gamma):
    c = Counter(gamma)
    return ((getz(c,0,0), [c[i] for i in c if i%2 == 0 and i > 0]),(0, [c[i] for i in c if i%2 == 1 and i > 0]))


def gamma2LusztigCellB(gamma):
    """
    This function returns the list of elements in the Lusztig cell of gamma
    """
    # Levi of even and odd parts
    Leven, Lodd = Wgamma(gamma)
    Ecell = Jind([sgnBC(Leven[0])],Leven[1])
    Ocell = Jind([sgnBC(Lodd[0])],Lodd[1])
    return(Ecell,Ocell)

In [109]:
tau = sgnBCD(6)
for t in Jind([tau],[5,5,5,5,5,5,5,5]):
    print(t)

([0, 1, 2, 3, 12, 13, 14], [1, 2, 3, 4, 13, 14])
([0, 1, 2, 3, 11, 13, 14], [1, 2, 3, 5, 13, 14])
([0, 1, 2, 3, 10, 13, 14], [1, 2, 3, 6, 13, 14])
([0, 1, 2, 3, 9, 13, 14], [1, 2, 3, 7, 13, 14])
([0, 1, 2, 3, 8, 13, 14], [1, 2, 3, 8, 13, 14])


In [110]:
print(gamma2LusztigCellB([40,2,0,60,40,2,0,8,60,40,2,0]))

([([0, 1, 4, 7], [1, 2, 6]), ([0, 1, 4, 6], [1, 2, 7]), ([0, 1, 3, 7], [1, 3, 6]), ([0, 1, 3, 6], [1, 3, 7])], [((0,), ())])
